# step B — RQ2 관측 (A2 생성): 어텐션·‖v‖·‖av‖

**대응 RQ:** RQ2 — 준수 실패를 매개하는 신호가 코드의 **형태**인가, 그 신호가 내부 어디에 있는가(관측). 인과는 step C.

**무엇을 재나** — 새 함수 **이름을 생성하는 바로 그 시점**의 query가, 선행 코드의 **이름 토큰**(하위 토큰 전부)과
지침 구간을 각각 얼마나 보는지(축 A), 그 그룹의 ‖v‖(축 B 크기)·‖av‖(축 A×B)를 층별로 잰다.
본문·`def`·괄호는 제외하고 **이름 식별자 토큰만** 잡는다.

**두 관측(view)**
- **View 1 — flip 2×2:** 선행 camel/snake **6/6 균형**, 지침 camel/snake 2종. 충돌 표기 토큰을 더 보는가
  (camel 지침→snake 토큰↑, snake 지침→camel 토큰↑). 지침 뒤집기로 '위반'을 만든다.
- **View 2 — 위반 개수 스윕:** 지침 camel 고정, 선행 camel **4/3/2/1/0** (stepA와 동일 격자; n=6은 View1). 위반↑에도 **지침 어텐션이 평탄**한가
  (평탄 → 실패가 '지침 결핍'이 아님).

설계 문서: `docs/stepB/plan.md`.

> **메모리(무료 T4):** Qwen2.5-Coder-3B fp16 ≈ 6.2GB. `output_attentions`로 전체 어텐션을 뜨지만 stepB 합성
> 프롬프트는 수백 토큰이라 감당된다(§2.7의 15GB/층은 16K 컨텍스트 얘기). 어텐션 가중치를 읽으려면 **eager**로 로드.
> **재개 가능:** 조건마다 즉시 저장, 이미 저장된 조건은 로드만. 끊겨도 셀 4·5·6을 다시 실행하면 이어서 한다.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin stepB/attention-observe
!git checkout stepB/attention-observe
!git pull --quiet origin stepB/attention-observe
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — 이 실험이 쓰는 조건 축 값 (View 1 flip + View 2 스윕)
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
SEEDS = list(range(20))     # 조건당 반복 (seed마다 다른 이름 12개)
REF_FRAC = 0.7              # 누적/요약 기준 층의 상대 위치

# (목표 표기, 선행 camel 개수) — 두 view를 합쳐 중복 제거
FLIP  = [(Notation.CAMEL, 6), (Notation.SNAKE, 6)]        # View1: 6/6 균형, 지침 flip
SWEEP = [(Notation.CAMEL, n) for n in (4, 3, 2, 1, 0)]     # View2: 지침 camel, 위반↑ (stepA 동일)
SPECS = list(dict.fromkeys(FLIP + SWEEP))                  # (camel,6)이 공유됨

def make(target, n, s):
    return Condition(
        model=MODEL,
        preceding=PrecedingCode(n_compliant=n, n_functions=12, composition=Composition.POOL),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=target),
        seed=s,
    )

conditions = [make(t, n, s) for (t, n) in SPECS for s in SEEDS]

# 실행 전 예측 (결과와 함께 보존, CLAUDE.md §6)
PREDICTION = ('View1: 충돌 표기 토큰을 더 봄(camel지침->snake, snake지침->camel), ‖v‖ 평탄. '
              'View2: 위반이 늘어도 지침 구간 어텐션은 평탄(실패=지침 결핍 아님).')
print(len(conditions), '조건 =', len(SPECS), 'spec x', len(SEEDS), 'seed')
print('specs (target, n_compliant):', [(t.value, n) for (t, n) in SPECS])

In [ ]:
# 실행 — 조건별 관측 + 즉시 저장(재개 가능). 중간중간 누적 결과를 요약 출력한다.
from collections import defaultdict
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

handle = load_model(MODEL, attn_implementation='eager')   # 어텐션 가중치 읽기 위해 eager
REF_LAYER = int(handle.num_layers * REF_FRAC)
print('layers:', handle.num_layers, '| GQA:', handle.gqa_info(), '| 기준층 L%d' % REF_LAYER)

acc = defaultdict(lambda: {'camel': [], 'snake': []})   # View1 누적(6/6 조건만): 지침별 토큰 어텐션
new = skipped = 0
for i, c in enumerate(conditions, 1):
    p = result_path(c, step='stepB')
    if p.exists():
        rec = load_result(p); skipped += 1
    else:
        out = run(c, handle=handle, mode='observe')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step='stepB', rq='RQ2', prediction=PREDICTION))
        rec = load_result(p); new += 1
    pl = rec.metrics.per_layer.get(REF_LAYER, {})
    n = rec.condition.preceding.n_compliant
    tgt = rec.condition.instruction.target_notation.value
    if n == 6 and 'code_camel__attention_weight' in pl and 'code_snake__attention_weight' in pl:
        acc[tgt]['camel'].append(pl['code_camel__attention_weight'])
        acc[tgt]['snake'].append(pl['code_snake__attention_weight'])
    if i % 20 == 0 or i == len(conditions):              # 중간 누적 출력
        print(f'[{i}/{len(conditions)}] 새 {new} / 건너뜀 {skipped}  (기준 L{REF_LAYER})')
        for t in ('camel', 'snake'):                     # View1: 충돌 표기를 더 보나(누적)
            cs, ss = acc[t]['camel'], acc[t]['snake']
            if cs:
                mc, ms = sum(cs)/len(cs), sum(ss)/len(ss)
                more = 'snake' if ms > mc else 'camel'
                print(f'    [View1] 지침={t}: camel토큰 {mc:.4f} vs snake토큰 {ms:.4f} '
                      f'-> {more} 더 봄 (n={len(cs)})')
print(f'완료: 새로 {new}, 건너뜀 {skipped}, 총 {len(conditions)}')

In [ ]:
# 결과 로드 — results/stepB/ 에 불변 저장된 이 실험 조건들을 모은다
from harness import result_path
from harness.results import load_result

records = [load_result(result_path(c, step='stepB')) for c in conditions]
print('로드:', len(records), '건 -> results/stepB/')

In [ ]:
# 요약 — View1(flip 2x2) + View2(위반 개수 스윕)
import pandas as pd, matplotlib.pyplot as plt

rows = []
for r in records:
    t = r.condition.instruction.target_notation.value
    n = r.condition.preceding.n_compliant
    tok = r.metrics.extra.get('span_token_counts', {})
    for layer, pl in r.metrics.per_layer.items():
        rows.append({'target': t, 'n_compliant': n, 'layer': int(layer),
                     'camel_attn': pl.get('code_camel__attention_weight'),
                     'snake_attn': pl.get('code_snake__attention_weight'),
                     'instr_attn': pl.get('instruction__attention_weight'),
                     'camel_av': pl.get('code_camel__av_norm'),
                     'snake_av': pl.get('code_snake__av_norm'),
                     'camel_v': pl.get('code_camel__v_norm'),
                     'snake_v': pl.get('code_snake__v_norm')})
df = pd.DataFrame(rows)
REF_LAYER = int((df.layer.max() + 1) * REF_FRAC)

# ── View 1 — flip 2x2 (선행 6/6) : 충돌 표기를 더 보는가 ──
bal = df[(df.n_compliant == 6) & (df.layer == REF_LAYER)]
tab = bal.groupby('target')[['camel_attn', 'snake_attn']].mean()
tab['더_본_쪽'] = np.where(tab['snake_attn'] > tab['camel_attn'], 'snake', 'camel')
print(f'[View1] flip 2x2 어텐션 @L{REF_LAYER} (충돌 표기를 더 보는가):'); print(tab.round(4))
print('  지침 구간 어텐션 (조건 간 평탄 기대):')
print(bal.groupby('target')['instr_attn'].mean().round(4))
print('  ‖v‖ (크기, 평탄 기대):')
print(bal.groupby('target')[['camel_v', 'snake_v']].mean().round(3))

# ── View 2 — 위반 개수 스윕 (지침=camel 고정) : 지침 어텐션 평탄한가 ──
# 지침 텍스트는 n_compliant와 무관하게 동일 → 지침 어텐션 합은 정규화 없이 비교 가능.
sw = df[(df.target == 'camel') & (df.layer == REF_LAYER)].groupby('n_compliant')
sweep_tab = pd.DataFrame({'지침_어텐션': sw['instr_attn'].mean(),
                          'snake(충돌)_av합': sw['snake_av'].mean()}).sort_index(ascending=False)
print(f'\n[View2] 위반 개수 스윕 @L{REF_LAYER} (n_compliant 6->0, 위반↑):'); print(sweep_tab.round(4))

# ── 플롯 ──
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
# 좌: View1 층별 궤적 (camel 지침일 때 camel vs snake 토큰 어텐션)
g = df[(df.n_compliant == 6) & (df.target == 'camel')].groupby('layer')[['camel_attn', 'snake_attn']].mean()
ax[0].plot(g.index, g['camel_attn'], marker='.', label='camel 토큰(준수)')
ax[0].plot(g.index, g['snake_attn'], marker='.', label='snake 토큰(충돌)')
ax[0].axvline(REF_LAYER, color='gray', ls='--', alpha=.5)
ax[0].set_title('View1: camel 지침, 층별 토큰 어텐션'); ax[0].set_xlabel('layer')
ax[0].set_ylabel('구간 어텐션 합(평균)'); ax[0].grid(True, alpha=.3); ax[0].legend()
# 우: View2 위반 개수별 지침 어텐션 (평탄 기대)
xs = sorted(df.n_compliant.unique(), reverse=True)
inst_by_n = [df[(df.target=='camel') & (df.n_compliant==n) & (df.layer==REF_LAYER)]['instr_attn'].mean() for n in xs]
ax[1].plot([str(n) for n in xs], inst_by_n, marker='o', color='C2')
ax[1].set_title('View2: 위반↑ 에도 지침 어텐션 평탄?'); ax[1].set_xlabel('n_compliant (6->0)')
ax[1].set_ylabel('지침 구간 어텐션'); ax[1].grid(True, alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# 밑줄/형태 마커 분석 — 형태 신호가 '어근 단어'가 아니라 '표층 마커'(_, 대문자)에 실리나.
# snake의 표식은 밑줄 '_', camel의 표식은 내부 대문자. 마커 토큰의 av가 어근보다 크면
# 모델이 내용이 아니라 형태를 읽는다는 직접 증거(RQ2 핵심).
REF = str(REF_LAYER)   # 셀 6에서 정한 기준 층
rows = []
for r in records:
    td = r.metrics.extra.get('token_detail', {})
    checks = [('code_snake', lambda s: '_' in s, '밑줄'),
              ('code_camel', lambda s: any(c.isupper() for c in s), '대문자')]
    for sp, is_marker, mlabel in checks:
        d = td.get(sp)
        if not d or not d.get('tokens'):
            continue
        arr = d['per_layer'].get(REF)
        if not arr:
            continue
        for tok, a, av in zip(d['tokens'], arr['a'], arr['av']):
            rows.append({'span': sp, 'text': tok['text'], 'a': a, 'av': av,
                         'kind': mlabel if is_marker(tok['text']) else '어근'})
det = pd.DataFrame(rows)
for sp in ['code_snake', 'code_camel']:
    sub = det[det.span == sp]
    if sub.empty:
        continue
    print(f'[{sp}] 형태 마커 vs 어근 토큰 (@L{REF}):')
    print(sub.groupby('kind')[['a', 'av']].mean().round(5))
    top = sub.sort_values('av', ascending=False)['text'].head(6).tolist()
    print('  av 상위 토큰:', top, '\n')

In [ ]:
# 결과 다운로드 — results/stepB 를 zip으로 묶어 내려받는다
import shutil
shutil.make_archive('stepB_results', 'zip', 'results/stepB')
try:
    from google.colab import files
    files.download('stepB_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): stepB_results.zip', e)